# Z2005 — Week 11: Graph Algorithms (Shortest Paths & Minimum Spanning Trees)

This notebook covers the core algorithms for finding shortest paths and minimum spanning trees on the weighted graphs introduced in Week 10: Dijkstra's algorithm, A* search, Bellman-Ford, and Prim's algorithm.


## Learning Objectives

By the end of this notebook you will be able to:

- Implement Dijkstra's algorithm from scratch, including reconstructing the actual shortest path (not just its length).
- Explain *why* Dijkstra's greedy strategy is correct for non-negative edge weights, and identify graphs where it produces the wrong answer.
- Implement A* search as a modification of Dijkstra's, and explain what an admissible heuristic is and why admissibility matters.
- Implement Bellman-Ford and explain why it tolerates negative edge weights while Dijkstra's does not, including detecting negative-weight cycles.
- Implement Prim's algorithm for minimum spanning trees and compare it against Kruskal's algorithm from Week 10 on the same graph.
- Choose the correct algorithm for a given graph problem (weighted/unweighted, negative weights or not, single-source shortest path vs. minimum spanning tree) and justify the choice.

## How to use this notebook

Run the cells from top to bottom. Cells marked `# TODO` are for you to fill in — they contain a function
signature and a docstring but raise `NotImplementedError` until you implement them. Cells with `assert`
statements are self-checks: they raise an `AssertionError` (or print an error) if your implementation is
wrong, and print a friendly success message and do nothing else if it is correct. Try every exercise
yourself before looking at the Solutions section at the end.

## 1. Dijkstra's Algorithm

Dijkstra's algorithm finds the shortest distance from a single source vertex to every other reachable
vertex in a weighted graph, provided every edge weight is non-negative. It works by repeatedly picking
the unvisited vertex with the smallest known tentative distance, "finalizing" that distance (it can never
improve later, because all remaining edges only add non-negative weight), and then *relaxing* its
outgoing edges: for each neighbor, checking whether reaching it through the current vertex is cheaper
than the best distance found so far.

You would reach for Dijkstra's whenever you need shortest paths in a graph with non-negative weights —
road networks with distances or travel times, network routing, or any "cheapest way to get from A to B"
problem where costs never go negative. The classic pitfall is running it on a graph that *does* have a
negative edge: Dijkstra's finalizes vertices greedily and never revisits them, so a later negative edge
that would have made an already-finalized vertex cheaper is simply never considered — see the counterexample
in the Bellman-Ford section below, and Section 3's MTech Extension in the MTech notebook.

We use `heapq` as a min-priority queue: the smallest tentative distance is always popped first, which is
exactly the "pick the closest unvisited vertex" step in $O(\log V)$ instead of the $O(V)$ scan a plain
list would require.

In [ ]:
import heapq

def dijkstra(adj: dict, start) -> tuple[dict, dict]:
    """Single-source shortest paths with Dijkstra's algorithm.

    adj: adjacency list, {node: [(neighbor, weight), ...]}, weights must be >= 0
    start: the source node

    Returns (dist, prev):
      dist[node] = shortest distance from start to node (absent if unreachable)
      prev[node] = the node visited immediately before `node` on that shortest
                   path, used later to reconstruct the actual route
    """
    dist = {start: 0}          # known shortest distance so far, updated as we relax edges
    prev = {start: None}       # predecessor pointers, for path reconstruction
    visited = set()            # vertices whose shortest distance is finalized
    pq = [(0, start)]          # min-heap of (tentative_distance, node)

    while pq:
        d, node = heapq.heappop(pq)   # always expand the closest unfinalized vertex next
        if node in visited:
            continue                  # a stale heap entry from an earlier, worse push
        visited.add(node)             # distance to `node` is now final and will not change
        for neighbor, weight in adj.get(node, []):
            if neighbor in visited:
                continue              # already finalized, nothing to gain by revisiting
            candidate = d + weight    # cost of reaching `neighbor` via `node`
            if candidate < dist.get(neighbor, float("inf")):
                dist[neighbor] = candidate   # found a cheaper route: relax the edge
                prev[neighbor] = node
                heapq.heappush(pq, (candidate, neighbor))
    return dist, prev


def reconstruct_path(prev: dict, target) -> list:
    """Walk the prev pointers backward from target to the source, then reverse."""
    path = []
    node = target
    while node is not None:
        path.append(node)
        node = prev[node]
    return path[::-1]


# Small worked example: three nodes, a direct edge 0->1 costing 10, and a
# cheaper detour 0->2->1 costing 1+1=2.
graph = {0: [(1, 10), (2, 1)], 1: [(0, 10), (2, 1)], 2: [(0, 1), (1, 1)]}
distances, predecessors = dijkstra(graph, 0)
print("distances from 0:", distances)
print("shortest path 0 -> 1:", reconstruct_path(predecessors, 1))
assert distances == {0: 0, 1: 2, 2: 1}          # the detour through 2 wins, not the direct edge
assert reconstruct_path(predecessors, 1) == [0, 2, 1]
print("Dijkstra's checks passed")

## 2. A* Search

A* is Dijkstra's algorithm with one addition: a *heuristic* function `h(node)` that estimates the
remaining distance to the goal, added to the priority-queue key. Instead of always expanding the vertex
closest to the *start* (Dijkstra's `f = g`), A* expands the vertex that looks most promising overall,
combining the distance travelled so far (`g`) with the estimated distance still to go (`h`), so
`f = g + h`. When the goal is known in advance — pathfinding on a map or a grid, for instance — this lets
A* skip large parts of the graph that Dijkstra's would have explored needlessly.

The heuristic must be *admissible*: it must never overestimate the true remaining distance. If it
overestimates, A* can commit to a finalized vertex too early and return a suboptimal path — the whole
correctness guarantee depends on this property, so it is worth checking explicitly rather than assuming
any "reasonable-looking" heuristic works. On a grid where you can move in the four cardinal directions,
Manhattan distance (`|dx| + |dy|`) is a standard admissible heuristic, because it can never overestimate
the number of unit steps required.

A common pitfall: using a heuristic that is admissible for diagonal movement (e.g. Euclidean distance)
on a grid that only allows cardinal moves — that heuristic is *not* guaranteed to underestimate correctly
in every graph, so always match the heuristic to the actual movement rules.

In [ ]:
def manhattan(a: tuple, b: tuple) -> int:
    """Admissible heuristic for a 4-directional grid: never overestimates true distance."""
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def grid_neighbors(node: tuple, walls: set, size: int = 3) -> list:
    """Cardinal (up/down/left/right) neighbors of `node` on a size x size grid, skipping walls."""
    x, y = node
    result = []
    for dx, dy in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
        nx, ny = x + dx, y + dy
        if 0 <= nx < size and 0 <= ny < size and (nx, ny) not in walls:
            result.append(((nx, ny), 1))   # every grid step costs 1
    return result


def a_star(start: tuple, goal: tuple, walls: set, size: int = 3) -> tuple[int, int]:
    """A* search on a grid. Returns (shortest_distance, number_of_nodes_explored)."""
    dist = {start: 0}                                  # g(node): cost from start
    visited = set()
    pq = [(manhattan(start, goal), start)]              # key is f = g + h; g(start) = 0
    explored = set()
    while pq:
        priority, node = heapq.heappop(pq)
        if node in visited:
            continue
        visited.add(node)
        explored.add(node)
        if node == goal:
            break                                       # goal reached: no need to keep searching
        for neighbor, weight in grid_neighbors(node, walls, size):
            new_g = dist[node] + weight
            if new_g < dist.get(neighbor, float("inf")):
                dist[neighbor] = new_g
                # f = g (actual cost so far) + h (estimated cost remaining)
                heapq.heappush(pq, (new_g + manhattan(neighbor, goal), neighbor))
    return dist.get(goal), len(explored)


def dijkstra_grid(start: tuple, goal: tuple, walls: set, size: int = 3) -> tuple[int, int]:
    """Same grid search, but plain Dijkstra's (h = 0 everywhere), for comparison."""
    dist = {start: 0}
    visited = set()
    pq = [(0, start)]
    explored = set()
    while pq:
        d, node = heapq.heappop(pq)
        if node in visited:
            continue
        visited.add(node)
        explored.add(node)
        if node == goal:
            break
        for neighbor, weight in grid_neighbors(node, walls, size):
            new_d = d + weight
            if new_d < dist.get(neighbor, float("inf")):
                dist[neighbor] = new_d
                heapq.heappush(pq, (new_d, neighbor))
    return dist.get(goal), len(explored)


walls = {(1, 1)}   # a single wall in the middle of a 3x3 grid
d_dist, d_explored = dijkstra_grid((0, 0), (2, 0), walls)
a_dist, a_explored = a_star((0, 0), (2, 0), walls)
print(f"Dijkstra's: distance={d_dist}, nodes explored={d_explored}")
print(f"A*:         distance={a_dist}, nodes explored={a_explored}")
assert d_dist == a_dist == 2          # both algorithms must agree on the shortest distance
assert a_explored <= d_explored       # A* should never explore more nodes than Dijkstra's here
print("A* checks passed: same distance, A* explores no more nodes than Dijkstra's")

## 3. Bellman-Ford

Bellman-Ford solves the same single-source shortest-path problem as Dijkstra's, but it works correctly
even when some edge weights are negative — something Dijkstra's greedy "finalize and never revisit"
strategy cannot handle. Instead of finalizing vertices one at a time in order of distance, Bellman-Ford
simply relaxes *every* edge in the graph, repeatedly, for `|V| - 1` rounds. That bound is not arbitrary:
any shortest path in a graph with no negative cycle visits at most `|V| - 1` edges (a path cannot usefully
repeat a vertex), so after `|V| - 1` rounds of relaxing every edge, every shortest distance is guaranteed
to have been found.

You would reach for Bellman-Ford specifically when negative weights are possible — for example, currency
arbitrage graphs where an edge weight is `-log(exchange rate)`, or any graph modeling a *cost that can
also be a gain*. The trade-off is speed: Bellman-Ford is $O(VE)$, while Dijkstra's with a binary heap is
$O((V+E)\log V)$, so use Dijkstra's whenever all weights are known to be non-negative.

A useful extra: if you run one more relaxation round after the `|V| - 1` guaranteed rounds and *any*
distance still improves, the graph contains a negative-weight cycle, and shortest paths are not
well-defined at all (you could loop the cycle forever, decreasing the "distance" without limit).

In [ ]:
def bellman_ford(edge_list: list, start, n: int) -> tuple[dict, dict]:
    """Single-source shortest paths, tolerant of negative edge weights.

    edge_list: [(u, v, weight), ...]
    start: source node
    n: total number of vertices (needed for the |V|-1 round bound)

    Returns (dist, prev), same meaning as in dijkstra().
    """
    dist = {start: 0}
    prev = {start: None}
    for _ in range(n - 1):                    # |V| - 1 rounds is provably enough
        for u, v, weight in edge_list:
            if u in dist and dist[u] + weight < dist.get(v, float("inf")):
                dist[v] = dist[u] + weight     # relax the edge u -> v
                prev[v] = u
    return dist, prev


def has_negative_cycle(edge_list: list, dist: dict) -> bool:
    """One extra relaxation round: if anything still improves, a negative cycle exists."""
    for u, v, weight in edge_list:
        if u in dist and dist[u] + weight < dist.get(v, float("inf")):
            return True
    return False


# A graph where the direct edge 0->1 looks cheap (cost 1), but a detour through
# a large negative edge (2->1, cost -10) is actually far cheaper: 0->2->1 = 4 + (-10) = -6.
neg_edges = [(0, 1, 1), (0, 2, 4), (2, 1, -10)]
dist, prev = bellman_ford(neg_edges, 0, 3)
print("Bellman-Ford distances:", dist)
assert dist == {0: 0, 1: -6, 2: 4}
assert reconstruct_path(prev, 1) == [0, 2, 1]
assert not has_negative_cycle(neg_edges, dist)

# Reproduce the same graph for Dijkstra's, to see it fail: it finalizes node 1 at
# distance 1 as soon as it is popped, and (correctly, by its own rules) never
# revisits a finalized node, so it never discovers the cheaper route through 2.
wrong_edges = {0: [(1, 1), (2, 4)], 1: [], 2: [(1, -10)]}
wrong_dist, _ = dijkstra(wrong_edges, 0)
print("Dijkstra's (wrong) distance to 1 on this same graph:", wrong_dist[1])
assert wrong_dist[1] == 1   # WRONG: the true shortest distance is -6, not 1

# Negative-cycle detection: 0->1->2->0 costs 1 + 1 + (-3) = -1, a genuine negative cycle.
cycle_edges = [(0, 1, 1), (1, 2, 1), (2, 0, -3)]
cycle_dist, _ = bellman_ford(cycle_edges, 0, 3)
assert has_negative_cycle(cycle_edges, cycle_dist) is True
print("Bellman-Ford checks passed: correct answer -6 where Dijkstra's wrongly reports 1")

## 4. Prim's Algorithm (Minimum Spanning Tree)

A minimum spanning tree (MST) connects every vertex of a weighted, undirected graph using the smallest
possible total edge weight, with no cycles. Week 10 covered Kruskal's algorithm, which builds the MST by
considering edges in increasing weight order and using a union-find structure to reject any edge that
would create a cycle. Prim's algorithm solves the same problem from a different angle: it grows a single
tree outward from a starting vertex, at every step adding the cheapest edge that connects the current tree
to a vertex not yet in it.

Both algorithms are greedy and both are provably correct (the "cut property" of MSTs guarantees that the
cheapest edge crossing any cut belongs to *some* MST), so for the same graph they always find an MST of
the same total weight — though not necessarily the same *set* of edges when weights tie. Prim's is
implemented here with the same min-heap pattern as Dijkstra's, and the two algorithms look almost
identical in code; the difference is what the priority queue key means. Dijkstra's key is *distance from
the start*; Prim's key is *cost of the single edge* connecting a candidate vertex to the tree so far. That
distinction matters: Prim's does not care how far a vertex is from the start overall, only how cheap the
next edge into the tree is.

Prim's tends to be preferred on dense graphs (many edges relative to vertices), because its cost is
$O(E \log V)$ with a binary heap and it never needs to sort the entire edge list up front; Kruskal's,
which does sort all edges first, is often simpler to reason about and preferred on sparse graphs.

In [ ]:
def prim(adj: dict, start, n: int) -> tuple[list, int]:
    """Minimum spanning tree via Prim's algorithm, grown outward from `start`.

    adj: adjacency list, {node: [(neighbor, weight), ...]}, undirected (both directions present)
    start: the vertex to grow the tree from (any vertex works, MST weight is the same)
    n: total number of vertices

    Returns (mst_edges, total_weight).
    """
    visited = {start}
    mst = []
    candidate_edges = [(weight, start, neighbor) for neighbor, weight in adj[start]]
    heapq.heapify(candidate_edges)   # min-heap of (edge_weight, from_node, to_node)
    total = 0
    while candidate_edges and len(visited) < n:
        weight, u, v = heapq.heappop(candidate_edges)   # cheapest edge leaving the tree so far
        if v in visited:
            continue                                    # both endpoints already in tree: would cycle
        visited.add(v)
        mst.append((u, v, weight))
        total += weight
        for neighbor, w in adj[v]:                       # tree just grew: add v's edges as candidates
            if neighbor not in visited:
                heapq.heappush(candidate_edges, (w, v, neighbor))
    return mst, total


def kruskal(n: int, edge_list: list) -> list:
    """Minimum spanning tree via Kruskal's algorithm (Week 10), for comparison.

    edge_list: [(weight, u, v), ...]
    """
    edge_list = sorted(edge_list)     # cheapest edges first
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x

    mst = []
    for weight, u, v in edge_list:
        if find(u) != find(v):        # adding this edge would not create a cycle
            mst.append((u, v, weight))
            parent[find(u)] = find(v)
    return mst


# Same 4-vertex graph used in Week 10, expressed for both algorithms.
mst_adj = {
    0: [(1, 1), (2, 3)],
    1: [(0, 1), (2, 2), (3, 5)],
    2: [(0, 3), (1, 2), (3, 4)],
    3: [(1, 5), (2, 4)],
}
prim_mst, prim_total = prim(mst_adj, 0, 4)
print("Prim's MST edges:", prim_mst, "total weight:", prim_total)
assert prim_total == 7

kruskal_edges = [(1, 0, 1), (2, 1, 2), (3, 0, 2), (4, 2, 3), (5, 1, 3)]
kruskal_mst = kruskal(4, kruskal_edges)
kruskal_total = sum(w for _, _, w in kruskal_mst)
print("Kruskal's MST edges:", kruskal_mst, "total weight:", kruskal_total)
assert kruskal_total == 7
print("Prim's checks passed: total weight 7, matching Kruskal's result on the identical graph")

## 5. A quick empirical comparison

The sections above establish correctness. It is worth also *seeing*, not just being told, that A*
explores fewer nodes than plain Dijkstra's on a larger grid with more walls, since a small 3x3 example is
too small to show much of a difference. We build a bigger grid with a maze-like wall pattern and time both
searches with `timeit`.

In [ ]:
import timeit

def big_walls(size: int) -> set:
    # A simple diagonal-ish wall pattern that forces some detouring, but leaves a path open.
    return {(x, y) for x in range(size) for y in range(size) if (x + y) % 4 == 0 and (x, y) not in {(0, 0), (size - 1, size - 1)}}

SIZE = 10
walls = big_walls(SIZE)
start, goal = (0, 0), (SIZE - 1, SIZE - 1)

d_dist, d_explored = dijkstra_grid(start, goal, walls, size=SIZE)
a_dist, a_explored = a_star(start, goal, walls, size=SIZE)
print(f"On a {SIZE}x{SIZE} grid: Dijkstra's explored {d_explored} nodes, A* explored {a_explored} nodes")
assert d_dist == a_dist                # both must still agree on the shortest distance
assert a_explored <= d_explored        # admissible heuristic guarantees A* never explores more

# Time both with the standard library's timeit, rather than asserting invented numbers.
dijkstra_time = timeit.timeit(lambda: dijkstra_grid(start, goal, walls, size=SIZE), number=200)
a_star_time = timeit.timeit(lambda: a_star(start, goal, walls, size=SIZE), number=200)
print(f"200 runs -- Dijkstra's: {dijkstra_time:.4f}s, A*: {a_star_time:.4f}s")

## 6. Floyd-Warshall (All-Pairs Shortest Paths)

Dijkstra's and Bellman-Ford both solve the *single-source* shortest-path problem: distances from one
start vertex to everywhere else. Floyd-Warshall instead solves the *all-pairs* problem in one pass:
shortest distances between **every** pair of vertices simultaneously. It is a dynamic programming
algorithm: `dist[i][j]` is refined by asking, for each intermediate vertex `k` in turn, "is it cheaper
to go from `i` to `j` by routing through `k`?" After considering every vertex as a possible intermediate
stop, `dist[i][j]` holds the true shortest distance.

The recurrence is:

$$dist_k[i][j] = \min\big(dist_{k-1}[i][j],\ dist_{k-1}[i][k] + dist_{k-1}[k][j]\big)$$

with `dist_0` initialized to the direct edge weights (infinity where no edge exists, zero on the
diagonal). Three nested loops over all vertices give $O(V^3)$ time and $O(V^2)$ space for the distance
matrix &mdash; worse than running Dijkstra's from every vertex ($O(VE\log V)$) on sparse graphs, but
simpler to implement and often faster in practice on dense graphs, and it handles negative edge weights
(though, like Bellman-Ford, not negative cycles &mdash; a negative cycle shows up as a negative value on
the diagonal, `dist[i][i] < 0`).

This is the same "consider one more resource/intermediate step and take the best of using it vs. not"
pattern you will meet again, in general form, in the Dynamic Programming week &mdash; Floyd-Warshall is
itself a textbook DP algorithm, just applied to graphs rather than sequences.

In [ ]:
def floyd_warshall(n: int, edge_list: list) -> list:
    """All-pairs shortest paths via dynamic programming.

    n: number of vertices, labeled 0..n-1
    edge_list: [(u, v, weight), ...], weights may be negative but the graph
               must not contain a negative-weight cycle

    Returns dist, an n x n matrix where dist[i][j] is the shortest distance
    from i to j (float('inf') if unreachable).
    """
    dist = [[float("inf")] * n for _ in range(n)]
    for i in range(n):
        dist[i][i] = 0                          # distance from a vertex to itself is 0
    for u, v, weight in edge_list:
        dist[u][v] = min(dist[u][v], weight)    # direct edges seed the matrix (keep the cheapest, if parallel)

    for k in range(n):              # k: the intermediate vertex we are now allowed to route through
        for i in range(n):
            for j in range(n):
                through_k = dist[i][k] + dist[k][j]
                if through_k < dist[i][j]:
                    dist[i][j] = through_k      # routing i -> k -> j beats the best route found so far
    return dist


# Small worked example: a 4-node graph, directed, with one negative edge (3 -> 0).
#
#        3 --(1)--> 2
#       /^           |
#    (7)|           (2)
#      \|            v
#        0 --(3)--> 1
#
fw_edges = [(0, 1, 3), (0, 3, 7), (1, 0, 8), (1, 2, 2), (2, 3, 1), (3, 0, 2), (3, 2, 5)]
fw_dist = floyd_warshall(4, fw_edges)
print("Floyd-Warshall distance matrix (rows = from, columns = to):")
for row in fw_dist:
    print(row)

# 0 -> 1 -> 2 = 3 + 2 = 5 beats the (nonexistent) direct edge
assert fw_dist[0][2] == 5
# 0 -> 1 -> 2 -> 3 = 3 + 2 + 1 = 6 beats the direct edge 0 -> 3 (weight 7)
assert fw_dist[0][3] == 6
# 1 -> 2 -> 3 = 2 + 1 = 3
assert fw_dist[1][3] == 3
# 3 -> 0 -> 1 = 2 + 3 = 5 beats going 3 -> 2 -> ... (no edge back to 1 that way)
assert fw_dist[3][1] == 5
print("Floyd-Warshall checks passed")


## 7. Cycle detection: where this already lives

Cycle detection was already covered in Week 10 (Introduction to Graphs), Section 6: undirected-graph
cycle detection via DFS with parent-tracking, and directed-graph cycle detection via the DFS three-color
(white/gray/black) scheme (see `has_cycle_undirected` in that notebook, and Exercise 4,
`find_cycle_directed`). It is worth pausing on *why* it matters here rather than re-deriving it: the
"one extra relaxation round" negative-cycle check in Section 3 above is exactly a cycle-detection
problem in disguise &mdash; Bellman-Ford relies on the fact that a *negative* cycle is the one situation
where shortest paths stop being well-defined, and the three-color DFS scheme from Week 10 is the general
tool for finding cycles (of any sign) in a directed graph when you need the actual cycle, not just a
yes/no answer.

## 8. Network Flow: Ford-Fulkerson and Max-Flow Min-Cut

A **flow network** is a directed graph where every edge has a **capacity**, plus a designated **source**
vertex (where flow originates) and **sink** vertex (where flow is collected). The question network flow
algorithms answer is: what is the largest total amount of "flow" that can be pushed from source to sink
without exceeding any edge's capacity? This models real problems directly &mdash; pipeline throughput,
network bandwidth, bipartite matching (job assignment, course scheduling), and traffic routing all reduce
to max-flow.

The **Ford-Fulkerson method** finds the max flow by repeatedly searching for an **augmenting path**: any
path from source to sink along which every edge still has spare capacity, pushes as much flow as the
tightest edge on that path allows (the **bottleneck**), and repeats until no augmenting path remains. The
key trick that makes this work even when an earlier choice of path was suboptimal is the **residual
graph**: alongside the remaining forward capacity on each edge, we track a reverse edge with capacity
equal to the flow already sent, so a later augmenting path can "undo" part of an earlier, worse choice by
sending flow backward along it.

Ford-Fulkerson as originally described leaves the choice of augmenting path unspecified, and a poor
choice (e.g., plain DFS) can be slow or even fail to terminate on irrational capacities. **Edmonds-Karp**
is the standard refinement: always find the augmenting path with the *fewest edges*, using BFS. This
guarantees termination in $O(VE^2)$ time regardless of capacity values, and is what we implement below.

The **max-flow min-cut theorem** guarantees that the value of the maximum flow exactly equals the
capacity of the smallest **cut** &mdash; the minimum total capacity of edges that, if removed, would
disconnect the source from the sink. This is not a coincidence: when Edmonds-Karp terminates (no more
augmenting paths), the set of vertices still reachable from the source in the residual graph, versus
everything else, *is* a minimum cut.

In [ ]:
from collections import deque

def bfs_augmenting_path(residual: dict, source, sink) -> list:
    """Find a shortest (fewest-edges) augmenting path from source to sink using BFS.

    residual: {node: {neighbor: remaining_capacity, ...}, ...}
    Returns the path as a list of vertices [source, ..., sink], or None if no
    augmenting path exists (source cannot reach sink with spare capacity).
    """
    parent = {source: None}
    queue = deque([source])
    while queue:
        node = queue.popleft()
        if node == sink:
            break
        for neighbor, cap in residual.get(node, {}).items():
            if cap > 0 and neighbor not in parent:     # unvisited edge with spare capacity
                parent[neighbor] = node
                queue.append(neighbor)
    if sink not in parent:
        return None                                    # sink unreachable: no augmenting path left
    path = []
    node = sink
    while node is not None:
        path.append(node)
        node = parent[node]
    return path[::-1]


def edmonds_karp(adj: dict, source, sink) -> int:
    """Maximum flow from source to sink (BFS-based Ford-Fulkerson).

    adj: adjacency list of capacities, {node: [(neighbor, capacity), ...]}
    Returns the value of the maximum flow.
    """
    residual = {}
    for u in adj:
        residual.setdefault(u, {})
        for v, cap in adj[u]:
            residual[u][v] = residual[u].get(v, 0) + cap   # forward capacity
            residual.setdefault(v, {})
            residual[v].setdefault(u, 0)                   # reverse edge, starts at 0 capacity

    max_flow = 0
    path = bfs_augmenting_path(residual, source, sink)
    while path is not None:
        bottleneck = min(residual[u][v] for u, v in zip(path, path[1:]))
        for u, v in zip(path, path[1:]):
            residual[u][v] -= bottleneck    # use up forward capacity
            residual[v][u] += bottleneck    # ...and open up an equal amount of reverse capacity
        max_flow += bottleneck
        path = bfs_augmenting_path(residual, source, sink)   # search again in the updated residual graph
    return max_flow


# Small worked example: source 0, sink 3.
#
#        (3)          (3)
#     0 -----> 1 -----> 3
#     |        |
#    (2)      (1)
#     |        v
#     +------> 2 -----> 3
#             (2)
#
# Iteration 1: augment along 0 -> 1 -> 3, bottleneck = min(3, 3) = 3.
# Iteration 2: augment along 0 -> 2 -> 3, bottleneck = min(2, 2) = 2.
# No augmenting path remains (0's edges are saturated): max flow = 3 + 2 = 5.
flow_net = {
    0: [(1, 3), (2, 2)],
    1: [(2, 1), (3, 3)],
    2: [(3, 2)],
    3: [],
}
result_flow = edmonds_karp(flow_net, 0, 3)
print("Max flow 0 -> 3:", result_flow)
assert result_flow == 5
print("Edmonds-Karp checks passed")


## Exercises

### Exercise 1: All shortest distances within a budget

Write `nodes_within_distance(adj, start, max_dist)` that returns the **set** of nodes reachable from
`start` with total distance less than or equal to `max_dist` (including `start` itself, at distance 0).
Reuse `dijkstra` rather than writing a new search from scratch.

Example:
```
adj = {0: [(1, 2), (2, 5)], 1: [(0, 2), (2, 2)], 2: [(0, 5), (1, 2)]}
nodes_within_distance(adj, 0, 3) -> {0, 1}   # node 2 is at distance 4 (0->1->2), too far
```

In [ ]:
def nodes_within_distance(adj: dict, start, max_dist) -> set:
    """Return the set of nodes reachable from `start` with shortest distance <= max_dist.

    adj: adjacency list, {node: [(neighbor, weight), ...]}, non-negative weights
    start: source node
    max_dist: inclusive distance budget
    """
    # TODO: implement this
    raise NotImplementedError

In [ ]:
adj1 = {0: [(1, 2), (2, 5)], 1: [(0, 2), (2, 2)], 2: [(0, 5), (1, 2)]}

result = nodes_within_distance(adj1, 0, 3)
assert result == {0, 1}, f"expected {{0, 1}}, got {result}"

result_all = nodes_within_distance(adj1, 0, 10)
assert result_all == {0, 1, 2}, f"expected all three nodes, got {result_all}"

result_only_start = nodes_within_distance(adj1, 0, 0)
assert result_only_start == {0}, f"expected just the start node, got {result_only_start}"
print("Exercise 1 passed")

### Exercise 2: Admissibility check

Write `is_admissible(heuristic, adj, goal, nodes)` that checks whether a given heuristic function never
overestimates the true shortest distance to `goal`, for every node in `nodes`. Use `dijkstra` to compute
the true distances (build a reversed adjacency list so distances are measured *to* `goal`, or simply run
Dijkstra's from `goal` if the graph is undirected — the graphs used for testing below are undirected, so
running Dijkstra's from `goal` directly gives the true distance from every node to `goal`).

Example:
```
adj = {0: [(1, 1)], 1: [(0, 1), (2, 1)], 2: [(1, 1)]}
h = lambda n: 0   # the zero heuristic is always admissible (it never overestimates anything)
is_admissible(h, adj, 2, [0, 1, 2]) -> True
```

In [ ]:
def is_admissible(heuristic, adj: dict, goal, nodes: list) -> bool:
    """Check heuristic(n) <= true shortest distance from n to goal, for every n in nodes.

    heuristic: a function node -> estimated distance to goal
    adj: undirected adjacency list, {node: [(neighbor, weight), ...]}
    goal: the target node
    nodes: the nodes to check the heuristic at
    """
    # TODO: implement this
    raise NotImplementedError

In [ ]:
adj2 = {0: [(1, 1)], 1: [(0, 1), (2, 1)], 2: [(1, 1)]}
zero_heuristic = lambda n: 0
# A heuristic that overestimates the distance from node 0 to node 2 (true distance is 2, this claims 100).
bad_heuristic = lambda n: 100 if n == 0 else 0
# The true distance function itself is always (trivially) admissible, since it equals, never exceeds, itself.
true_dist_from_2, _ = dijkstra(adj2, 2)
exact_heuristic = lambda n: true_dist_from_2.get(n, float("inf"))

assert is_admissible(zero_heuristic, adj2, 2, [0, 1, 2]) is True
assert is_admissible(bad_heuristic, adj2, 2, [0, 1, 2]) is False
assert is_admissible(exact_heuristic, adj2, 2, [0, 1, 2]) is True
print("Exercise 2 passed")

### Exercise 3: Cheapest flight with at most k stops

Write `cheapest_with_stops(edge_list, n, start, goal, k)` that finds the cheapest cost from `start` to
`goal` using **at most `k` edges** (stops), where edge weights may be negative. This is a Bellman-Ford
variant: instead of running `n - 1` full rounds, run only `k` rounds, since each round only ever extends
paths by exactly one more edge. Return `float('inf')` if `goal` is unreachable within `k` edges.

Example:
```
edges = [(0, 1, 100), (0, 2, 500), (1, 2, 100)]
cheapest_with_stops(edges, 3, 0, 2, 1) -> 500   # direct 0->2 only, 1 edge allowed
cheapest_with_stops(edges, 3, 0, 2, 2) -> 200   # 0->1->2 is cheaper, needs 2 edges
```

In [ ]:
def cheapest_with_stops(edge_list: list, n: int, start, goal, k: int) -> float:
    """Cheapest cost from start to goal using at most k edges (Bellman-Ford limited to k rounds).

    edge_list: [(u, v, weight), ...], weights may be negative
    n: number of vertices
    start, goal: source and target
    k: maximum number of edges allowed in the path
    """
    # TODO: implement this
    raise NotImplementedError

In [ ]:
flights = [(0, 1, 100), (0, 2, 500), (1, 2, 100)]
assert cheapest_with_stops(flights, 3, 0, 2, 1) == 500
assert cheapest_with_stops(flights, 3, 0, 2, 2) == 200

unreachable = cheapest_with_stops(flights, 3, 2, 0, 5)
assert unreachable == float("inf")

start_equals_goal = cheapest_with_stops(flights, 3, 0, 0, 0)
assert start_equals_goal == 0
print("Exercise 3 passed")

### Exercise 4: Second-cheapest spanning tree edge set (harder)

Write `mst_is_unique(n, edge_list)` that returns `True` if the graph has exactly one minimum spanning
tree (by total weight *and* by edge set), and `False` if there are multiple distinct MSTs with the same
minimum total weight. Hint: build the MST once with Kruskal's, note its total weight, then check whether
any edge *not* in that MST could be swapped in for an MST edge of the same weight to form a different
valid spanning tree of the same total weight.

Example:
```
edges = [(1, 0, 1), (2, 1, 2), (2, 0, 2)]
# 0-1 (weight 1), then either 1-2 or 0-2 (both weight 2) complete the MST at total weight 3: not unique
mst_is_unique(3, edges) -> False
```

In [ ]:
def mst_is_unique(n: int, edge_list: list) -> bool:
    """Return True iff the minimum spanning tree of this graph is unique.

    edge_list: [(weight, u, v), ...]
    """
    # TODO: implement this
    raise NotImplementedError

In [ ]:
tied_edges = [(1, 0, 1), (2, 1, 2), (2, 0, 2)]
unique_edges = [(1, 0, 1), (2, 1, 2), (5, 0, 2), (10, 1, 3)]
single_edge_graph = [(7, 0, 1)]

assert mst_is_unique(3, tied_edges) is False   # two equal-weight edges (0-2 and 1-2) are interchangeable
assert mst_is_unique(4, unique_edges) is True   # all weights distinct enough that no swap ties the total
assert mst_is_unique(2, single_edge_graph) is True
print("Exercise 4 passed")

### Exercise 5: Graph diameter with Floyd-Warshall

Write `graph_diameter(n, edge_list)` that returns the **diameter** of a graph: the largest shortest-path
distance between any pair of distinct vertices. Use `floyd_warshall` to get all pairwise distances first.
If any pair of vertices is unreachable from each other, the diameter is undefined; return `float('inf')`
in that case.

Example:
```
edges = [(0, 1, 1), (1, 0, 1), (1, 2, 1), (2, 1, 1), (2, 3, 1), (3, 2, 1)]
graph_diameter(4, edges) -> 3   # farthest pair is 0 and 3, distance 3
```

In [ ]:
def graph_diameter(n: int, edge_list: list) -> float:
    """Largest shortest-path distance between any pair of distinct vertices.

    n: number of vertices
    edge_list: [(u, v, weight), ...]
    """
    # TODO: implement this
    raise NotImplementedError


In [ ]:
diam_edges = [(0, 1, 1), (1, 0, 1), (1, 2, 1), (2, 1, 1), (2, 3, 1), (3, 2, 1)]
assert graph_diameter(4, diam_edges) == 3

disconnected_edges = [(0, 1, 1), (1, 0, 1)]
assert graph_diameter(3, disconnected_edges) == float("inf")
print("Exercise 5 passed")


### Exercise 6: Which vertices are corrupted by a negative cycle?

Write `vertices_reachable_from_negative_cycle(edge_list, n, start)` that returns the **set** of vertices
whose shortest distance from `start` is not well-defined because a negative-weight cycle lies on some
path to them. Run Bellman-Ford for the usual `n - 1` rounds, then do **one more round**: any vertex whose
distance still improves in that extra round is directly corrupted; then propagate outward (a vertex
reachable *from* a corrupted vertex is also corrupted, since you could route through the cycle first to
make its distance arbitrarily small too).

Example:
```
edges = [(0, 1, 1), (1, 2, 1), (2, 1, -3), (2, 3, 1)]
# 1 -> 2 -> 1 is a cycle of weight 1 + (-3) = -2, a negative cycle
vertices_reachable_from_negative_cycle(edges, 4, 0) -> {1, 2, 3}
```

In [ ]:
def vertices_reachable_from_negative_cycle(edge_list: list, n: int, start) -> set:
    """Vertices whose shortest distance from start is corrupted by a negative cycle.

    edge_list: [(u, v, weight), ...]
    n: number of vertices
    start: source node
    """
    # TODO: implement this
    raise NotImplementedError


In [ ]:
nc_edges = [(0, 1, 1), (1, 2, 1), (2, 1, -3), (2, 3, 1)]
assert vertices_reachable_from_negative_cycle(nc_edges, 4, 0) == {1, 2, 3}

clean_edges = [(0, 1, 1), (1, 2, 1)]
assert vertices_reachable_from_negative_cycle(clean_edges, 3, 0) == set()
print("Exercise 6 passed")


### Exercise 7: Find the minimum cut edges (harder)

Write `min_cut_edges(adj, source, sink)` that returns the **set** of edges `(u, v)` crossing a minimum
cut, after running max flow to completion. By the max-flow min-cut theorem, once `edmonds_karp` has
saturated the network, the set of vertices still reachable from `source` in the final residual graph
(via BFS, following only edges with remaining capacity > 0) forms one side of a minimum cut; the crossing
edges are exactly the original edges that go from a reachable vertex to an unreachable one. You will need
to reuse the residual-graph construction and augmenting-path loop from `edmonds_karp`.

Example:
```
flow_net = {0: [(1, 3), (2, 2)], 1: [(2, 1), (3, 3)], 2: [(3, 2)], 3: []}
min_cut_edges(flow_net, 0, 3) -> {(0, 1), (0, 2)}   # total capacity 3 + 2 = 5, matches the max flow
```

In [ ]:
def min_cut_edges(adj: dict, source, sink) -> set:
    """Edges crossing a minimum cut between source and sink.

    adj: adjacency list of capacities, {node: [(neighbor, capacity), ...]}
    """
    # TODO: implement this
    raise NotImplementedError


In [ ]:
flow_net = {0: [(1, 3), (2, 2)], 1: [(2, 1), (3, 3)], 2: [(3, 2)], 3: []}
cut = min_cut_edges(flow_net, 0, 3)
assert cut == {(0, 1), (0, 2)}

cut_capacity = sum(cap for u, v_cap_list in flow_net.items() for v, cap in v_cap_list if (u, v) in cut)
assert cut_capacity == 5   # equals the max flow value, by the max-flow min-cut theorem
print("Exercise 7 passed")


## Quiz

**1. A graph has one edge with weight -1 and all other edges non-negative. Which algorithm should you use
to guarantee correct shortest paths, and why?**

<details><summary>Show answer</summary>
Bellman-Ford. Dijkstra's finalizes each vertex's distance as soon as it is popped from the priority queue
and never revisits a finalized vertex; if a negative edge later offers a cheaper route to an already-
finalized vertex, Dijkstra's will never find it. Bellman-Ford relaxes every edge repeatedly and makes no
such "finalize and forget" assumption, so it remains correct with negative weights (as long as there is
no negative cycle).
</details>

**2. You are pathfinding on a grid where diagonal moves are allowed and cost 1 (same as cardinal moves).
Is Manhattan distance still an admissible heuristic?**

<details><summary>Show answer</summary>
No. Manhattan distance can overestimate the true cost once diagonal moves are allowed, because a diagonal
move covers what Manhattan distance counts as two units of movement for the cost of one. An admissible
heuristic here would be Chebyshev distance, `max(|dx|, |dy|)`, which never overestimates the number of
moves needed when diagonals are free 1-cost moves.
</details>

**3. Kruskal's algorithm sorts every edge before it starts. Prim's algorithm never sorts the full edge
list. Does this mean Prim's is always faster?**

<details><summary>Show answer</summary>
Not always. Prim's avoids an upfront full sort, but with a binary heap it is still `O(E log V)`, the same
asymptotic complexity as Kruskal's `O(E log E)` (and `log E` and `log V` are within a constant factor of
each other since `E` is at most `V^2`). In practice, Prim's tends to have an edge on dense graphs (many
edges per vertex) since it never needs the whole edge list sorted at once, while Kruskal's is often simpler
to reason about and preferred on sparse graphs. Neither dominates the other in every case.
</details>

**4. True or false: if all edge weights in a graph are distinct, the minimum spanning tree is guaranteed
to be unique.**

<details><summary>Show answer</summary>
True. With all-distinct weights there is never a tie to break when choosing the next edge (in either
Kruskal's or Prim's), so the greedy choice at every step is forced, and both algorithms must produce the
same MST. Ties in edge weight are exactly what create the possibility of multiple valid MSTs, as explored
in Exercise 4.
</details>

**5. You need all-pairs shortest paths on a graph with 50 vertices and 2,000 edges (fairly dense).
Would you run Dijkstra's from every vertex, or use Floyd-Warshall?**

<details><summary>Show answer</summary>
Either works, but they trade off differently. Running Dijkstra's from all 50 vertices costs
`O(V * E log V) = O(50 * 2000 * log 50) ~ 566,000` operations. Floyd-Warshall costs `O(V^3) = 125,000`
operations here, and is simpler to implement (no priority queue, no per-source bookkeeping) &mdash; a
reasonable choice at this density. Floyd-Warshall is generally preferred when the graph is dense enough
that `E` approaches `V^2`, or when negative edges are present (Dijkstra's from every vertex would need to
be replaced by Bellman-Ford from every vertex, at `O(V^2 E)`, far worse than Floyd-Warshall's `O(V^3)`).
</details>

**6. In the residual graph used by Edmonds-Karp, why does every edge need a reverse edge, even one that
did not exist in the original network?**

<details><summary>Show answer</summary>
The reverse edge lets a later augmenting path "undo" flow sent by an earlier, suboptimal choice. If the
algorithm first sends flow along a path that turns out not to be part of the true maximum flow, pushing
flow backward along the reverse edge effectively reroutes that flow elsewhere without needing to search
for it explicitly. Without reverse edges, Ford-Fulkerson could get stuck at a flow value lower than the
true maximum, because it would have no way to reconsider an earlier commitment.
</details>

## Solutions (try the exercises yourself first!)

**Exercise 1 solution:**

In [ ]:
def nodes_within_distance_solution(adj: dict, start, max_dist) -> set:
    dist, _ = dijkstra(adj, start)               # reuse Dijkstra's for all shortest distances
    return {node for node, d in dist.items() if d <= max_dist}

# Overwrite the TODO version so the self-check cell above can be re-run against the real solution.
nodes_within_distance = nodes_within_distance_solution

result = nodes_within_distance(adj1, 0, 3)
assert result == {0, 1}
print("Exercise 1 solution verified")

**Exercise 2 solution:**

In [ ]:
def is_admissible_solution(heuristic, adj: dict, goal, nodes: list) -> bool:
    true_dist, _ = dijkstra(adj, goal)            # true distance FROM goal == distance TO goal (undirected)
    for node in nodes:
        if node == goal:
            true_d = 0
        else:
            true_d = true_dist.get(node, float("inf"))
        if heuristic(node) > true_d:              # overestimate found: not admissible
            return False
    return True

is_admissible = is_admissible_solution

assert is_admissible(zero_heuristic, adj2, 2, [0, 1, 2]) is True
assert is_admissible(bad_heuristic, adj2, 2, [0, 1, 2]) is False
print("Exercise 2 solution verified")

**Exercise 3 solution:**

In [ ]:
def cheapest_with_stops_solution(edge_list: list, n: int, start, goal, k: int) -> float:
    dist = {start: 0}
    for _ in range(k):                            # exactly k rounds: at most k edges used
        updated = dict(dist)                      # relax using distances as of the START of this round,
        for u, v, weight in edge_list:             # so we never chain two relaxations within one round
            if u in dist and dist[u] + weight < updated.get(v, float("inf")):
                updated[v] = dist[u] + weight
        dist = updated
    return dist.get(goal, float("inf"))

cheapest_with_stops = cheapest_with_stops_solution

assert cheapest_with_stops(flights, 3, 0, 2, 1) == 500
assert cheapest_with_stops(flights, 3, 0, 2, 2) == 200
print("Exercise 3 solution verified")

**Exercise 4 solution:**

In [ ]:
def mst_is_unique_solution(n: int, edge_list: list) -> bool:
    sorted_edges = sorted(edge_list)
    mst_edges = kruskal(n, [(w, u, v) for w, u, v in sorted_edges])
    mst_total = sum(w for _, _, w in mst_edges)
    mst_edge_set = {frozenset((u, v)) for u, v, w in mst_edges}

    # Try every edge NOT in the MST: if adding it and dropping some MST edge of the
    # same weight still connects everything at the same total weight, the MST is not unique.
    for weight, u, v in sorted_edges:
        if frozenset((u, v)) in mst_edge_set:
            continue
        for mu, mv, mw in mst_edges:
            if mw != weight:
                continue
            # Swap (mu, mv) out, (u, v) in, and check the result is still a spanning tree.
            candidate = [e for e in mst_edges if not (e[0] == mu and e[1] == mv)]
            candidate.append((u, v, weight))
            parent = list(range(n))

            def find(x, parent=parent):
                while parent[x] != x:
                    x = parent[x]
                return x

            ok = True
            for cu, cv, cw in candidate:
                ru, rv = find(cu), find(cv)
                if ru == rv:
                    ok = False
                    break
                parent[ru] = rv
            if ok and len(candidate) == n - 1:
                return False   # found a genuinely different MST of the same total weight
    return True

mst_is_unique = mst_is_unique_solution

assert mst_is_unique(3, tied_edges) is False
assert mst_is_unique(4, unique_edges) is True
print("Exercise 4 solution verified")

**Exercise 5 solution:**

In [ ]:
def graph_diameter_solution(n: int, edge_list: list) -> float:
    dist = floyd_warshall(n, edge_list)
    pairwise = [dist[i][j] for i in range(n) for j in range(n) if i != j]
    if any(d == float("inf") for d in pairwise):
        return float("inf")               # some pair is unreachable: diameter undefined
    return max(pairwise)

graph_diameter = graph_diameter_solution

assert graph_diameter(4, diam_edges) == 3
assert graph_diameter(3, disconnected_edges) == float("inf")
print("Exercise 5 solution verified")


**Exercise 6 solution:**

In [ ]:
def vertices_reachable_from_negative_cycle_solution(edge_list: list, n: int, start) -> set:
    dist = {start: 0}
    for _ in range(n - 1):                        # the usual n - 1 rounds
        for u, v, w in edge_list:
            if u in dist and dist[u] + w < dist.get(v, float("inf")):
                dist[v] = dist[u] + w

    corrupted = set()
    for u, v, w in edge_list:                     # one extra round: anything that still improves is corrupted
        if u in dist and dist[u] + w < dist.get(v, float("inf")):
            corrupted.add(v)

    changed = True
    while changed:                                 # propagate corruption forward along the graph
        changed = False
        for u, v, w in edge_list:
            if u in corrupted and v not in corrupted:
                corrupted.add(v)
                changed = True
    return corrupted

vertices_reachable_from_negative_cycle = vertices_reachable_from_negative_cycle_solution

assert vertices_reachable_from_negative_cycle(nc_edges, 4, 0) == {1, 2, 3}
assert vertices_reachable_from_negative_cycle(clean_edges, 3, 0) == set()
print("Exercise 6 solution verified")


**Exercise 7 solution:**

In [ ]:
def min_cut_edges_solution(adj: dict, source, sink) -> set:
    residual = {}
    for u in adj:
        residual.setdefault(u, {})
        for v, cap in adj[u]:
            residual[u][v] = residual[u].get(v, 0) + cap
            residual.setdefault(v, {})
            residual[v].setdefault(u, 0)

    path = bfs_augmenting_path(residual, source, sink)
    while path is not None:                        # run max flow to completion first
        bottleneck = min(residual[u][v] for u, v in zip(path, path[1:]))
        for u, v in zip(path, path[1:]):
            residual[u][v] -= bottleneck
            residual[v][u] += bottleneck
        path = bfs_augmenting_path(residual, source, sink)

    reachable = {source}                            # BFS in the final residual graph from source
    queue = deque([source])
    while queue:
        node = queue.popleft()
        for neighbor, cap in residual.get(node, {}).items():
            if cap > 0 and neighbor not in reachable:
                reachable.add(neighbor)
                queue.append(neighbor)

    cut_edges = set()
    for u in adj:
        for v, cap in adj[u]:
            if u in reachable and v not in reachable:
                cut_edges.add((u, v))
    return cut_edges

min_cut_edges = min_cut_edges_solution

cut = min_cut_edges(flow_net, 0, 3)
assert cut == {(0, 1), (0, 2)}
print("Exercise 7 solution verified")
